# PolyChain v27: Polymer Property Prediction
**ANRF AISEHack 2.0 - Polymer Property Prediction**

8-model ensemble: XGBoost, LightGBM, CatBoost, Random Forest, MLP, GCN, GAT, MPNN

Run all cells in order. Pipeline skips completed work on re-run.

In [ ]:
# Cell 1: Install dependencies
import sys, os, subprocess, shutil, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print(f"Python: {sys.version[:6]}")
print(f"Executable: {sys.executable}")

# Detect GPU
def detect_gpu():
    try:
        out = subprocess.check_output(
            "nvidia-smi --query-gpu=name --format=csv,noheader",
            shell=True, timeout=10
        ).decode().strip()
        return out[:50]
    except Exception:
        return "CPU"

gpu = detect_gpu()
print(f"Device: {gpu}")

# Install PyTorch if needed
try:
    import torch
    print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
except ImportError:
    print("Installing PyTorch...")
    if "P100" in gpu:
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "torch==2.5.1+cu121", "--index-url",
                        "https://download.pytorch.org/whl/cu121", "-q"],
                       capture_output=True)
    elif "T4" in gpu:
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "torch==2.6.0+cu124", "--index-url",
                        "https://download.pytorch.org/whl/cu124", "-q"],
                       capture_output=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch", "-q"],
                       capture_output=True)
    import torch
    print(f"PyTorch {torch.__version__}")

# Install PyG
pyg_check = subprocess.run([sys.executable, "-c", "import torch_geometric"],
                           capture_output=True)
if pyg_check.returncode != 0:
    print("Installing PyTorch Geometric...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"],
                   capture_output=True)

# Install other deps
for pkg in ["rdkit", "xgboost", "lightgbm", "catboost", "optuna",
            "scikit-learn", "pandas", "numpy", "pyyaml"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True)

print("All dependencies ready!")

In [ ]:
# Cell 2: Clone repo and setup
WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/NotShubham1112/Poly.git"
BRANCH = "feat/competition-data-adaptation"

os.chdir(WORK_DIR)
REPO_DIR = os.path.join(WORK_DIR, "Poly")

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.check_call(["git", "checkout", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(["git", "pull", "origin", BRANCH],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.chdir(WORK_DIR)
else:
    subprocess.check_call([
        "git", "clone", "-b", BRANCH, "--single-branch", REPO_URL
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Copy polymer_competition contents to work dir
SRC = os.path.join(REPO_DIR, "polymer_competition")
for item in os.listdir(SRC):
    src_path = os.path.join(SRC, item)
    dst_path = os.path.join(WORK_DIR, item)
    if os.path.exists(dst_path):
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path)
    else:
        shutil.copy2(src_path, dst_path)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {len(os.listdir('.'))} items")

In [ ]:
# Cell 3: Build features and CV splits
!python -m data.generate_splits --config config.yaml --target tg
!python -m data.generate_splits --config config.yaml --target egc
!python -m features.build_features --config config.yaml
print("Features and CV splits ready!")

In [ ]:
# Cell 4: Train all models via pipeline
# This runs: splits -> features -> train -> ensemble -> reports
# Completed folds are auto-skipped on re-run
!python generate_all.py --steps 3,4 --models xgb,lgb,catboost,rf,mlp,gcn,gat,mpnn --config config.yaml
print("Training complete!")

In [ ]:
# Cell 5: Build ensemble and generate submission
!python -m ensemble.build_ensemble --config config.yaml --target tg
!python -m ensemble.build_ensemble --config config.yaml --target egc
!python -m data.merge_submissions --config config.yaml
print("Ensemble and submission ready!")

In [ ]:
# Cell 6: Verify submission and copy to output
import pandas as pd
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

ver = cfg["experiment"]["version"]
sub_path = f"outputs/submissions/submission.csv"

df = pd.read_csv(sub_path)
print(f"Submission: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"ID range: [{df['id'].min()}, {df['id'].max()}]")
print(f"Target range: [{df['target'].min():.2f}, {df['target'].max():.2f}]")
print(f"Target mean: {df['target'].mean():.2f}")
print()
print(df.head(10))

# Copy to Kaggle output
import shutil
shutil.copy(sub_path, "/kaggle/working/submission.csv")
print(f"\nCopied to /kaggle/working/submission.csv")

In [ ]:
# Cell 7: Show model performance summary
import json
from pathlib import Path

manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    
    # Group by model_type and target
    from collections import defaultdict
    results = defaultdict(lambda: defaultdict(list))
    for entry in data:
        if entry.get("status") == "completed":
            model = entry.get("model_type", "?")
            target = entry.get("target", "?")
            score = entry.get("score", 0)
            results[model][target].append(score)
    
    print(f"{'Model':<18} {'TG R²':>10} {'EGC R²':>10} {'Mean R²':>10}")
    print("-" * 50)
    for model in sorted(results.keys()):
        tg_scores = results[model].get("tg", [0])
        egc_scores = results[model].get("egc", [0])
        tg_mean = sum(tg_scores) / len(tg_scores) if tg_scores else 0
        egc_mean = sum(egc_scores) / len(egc_scores) if egc_scores else 0
        mean_r2 = (tg_mean + egc_mean) / 2
        print(f"{model:<18} {tg_mean:>10.4f} {egc_mean:>10.4f} {mean_r2:>10.4f}")
else:
    print("No experiment manifest found.")